# PCT Knowledge Distillation Training

Trains a leaner PCT model from the Point-Transformers implementation: https://github.com/qq456cvb/Point-Transformers

Note: specifically for training it on full PAPNet data.

## Env prep

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Paths, folders
import os

REPO_PATH = '/content/pointcloud-bench'
datasets_folder_path = '/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/datasets'
output_path = '/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/results'
checkpoints_folder_path = '/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/checkpoints'
logs_path = os.path.join(output_path, "pct_logs")

if not os.path.exists(logs_path):
  os.makedirs(logs_path, exist_ok=True)

In [ ]:
# Get repo
!git clone --recurse-submodules --branch lean-model https://github.com/DavidClaszen/pointcloud-bench {REPO_PATH}

# Submodule handling
%cd {REPO_PATH}
!git submodule update --init --recursive

Cloning into '/content/pointcloud-bench'...
remote: Enumerating objects: 633, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 633 (delta 65), reused 106 (delta 30), pack-reused 465 (from 2)
Receiving objects: 100% (633/633), 52.81 MiB | 15.78 MiB/s, done.
Resolving deltas: 100% (270/270), done.
Submodule 'repos/PAPNet' (https://github.com/DavidClaszen/PAPNet.git) registered for path 'repos/PAPNet'
Submodule 'repos/Point-Transformers' (https://github.com/DavidClaszen/Point-Transformers.git) registered for path 'repos/Point-Transformers'
Cloning into '/content/pointcloud-bench/repos/PAPNet'...
remote: Enumerating objects: 145, done.        
remote: Counting objects: 100% (145/145), done.        
remote: Compressing objects: 100% (117/117), done.        
remote: Total 145 (delta 63), reused 88 (delta 27), pack-reused 0 (from 0)        
Receiving objects: 100% (145/145), 9.66 MiB | 9.77 MiB/s, done.
Resolving deltas: 10

In [ ]:
# Update Submodule
%cd {REPO_PATH}/repos/Point-Transformers/
!git fetch origin lean-model
!git checkout lean-model
!git pull origin lean-model
%cd {REPO_PATH}

/content/pointcloud-bench/repos/Point-Transformers
From https://github.com/DavidClaszen/Point-Transformers
 * branch            lean-model -> FETCH_HEAD
Branch 'lean-model' set up to track remote branch 'lean-model' from 'origin'.
Switched to a new branch 'lean-model'
From https://github.com/DavidClaszen/Point-Transformers
 * branch            lean-model -> FETCH_HEAD
Already up to date.
/content/pointcloud-bench


In [ ]:
# Install Requirements
%pip install -r {REPO_PATH}/envs/pct/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.7 MB/s eta 0:00:00


In [ ]:
# Check for CUDA/GPU
import torch, sys
print(sys.version)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.is_available())

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.9.0+cu126 CUDA: 12.6 GPU: True


In [ ]:
# Copy and unzip only the partialmodelnet40 set
# Evaluation will be done in other notebook
!mkdir /content/downloads
!rsync -avP {datasets_folder_path}/partialmodelnet40.tar.gz /content/downloads
!tar -xvzf /content/downloads/partialmodelnet40.tar.gz -C {REPO_PATH}/datasets

sending incremental file list
partialmodelnet40.tar.gz
  2,162,130,396 100%   63.93MB/s    0:00:32 (xfr#1, to-chk=0/1)

sent 2,162,658,366 bytes  received 35 bytes  64,556,967.19 bytes/sec
total size is 2,162,130,396  speedup is 1.00
partialmodelnet40/
partialmodelnet40/test_labels.npy
partialmodelnet40/test_gt_tra.npy
partialmodelnet40/test_gt_rot.npy
partialmodelnet40/partialmodelnet40_shape_names.txt
partialmodelnet40/partialmodelnet40_test.txt
partialmodelnet40/partialmodelnet40_train.txt
partialmodelnet40/train_points.npy
partialmodelnet40/train_labels.npy
partialmodelnet40/train_gt_tra.npy
partialmodelnet40/train_gt_rot.npy
partialmodelnet40/test_points.npy


# Model Training

Since we're only using PAPNet style data here, always set `use_papnet_loader` to `True`.


In [ ]:
!git pull
!git submodule update

remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 13 (delta 11), reused 13 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 1.18 KiB | 605.00 KiB/s, done.
From https://github.com/DavidClaszen/Point-Transformers
   b408f7a..da6faf6  lean-model -> origin/lean-model
Updating b408f7a..da6faf6
Fast-forward
 config/kd.yaml                           | 2 +-
 config/model/lean_minimal.yaml           | 1 +
 config/model/lean_set_decoder_lbrd1.yaml | 1 +
 config/model/lean_set_sa_ch_128.yaml     | 1 +
 config/model/lean_set_sa_ch_64.yaml      | 1 +
 config/model/lean_set_sa_layer1.yaml     | 1 +
 config/model/lean_set_sa_stacks1.yaml    | 1 +
 config/model/lean_set_sa_stacks2.yaml    | 1 +
 config/model/lean_set_sa_stacks3.yaml    | 1 +
 9 files changed, 9 insertions(+), 1 deletion(-)


In [ ]:
# Train Lean
# At the end of this code, the logs as well as the best performing models for each experiment will be saved
%cd {REPO_PATH}/repos/Point-Transformers
!mkdir {checkpoints_folder_path}/pct_lean
!python train_kd.py --multirun \
model=lean_minimal,lean_set_decoder_lbrd1,lean_set_sa_ch_64,lean_set_sa_ch_128,lean_set_sa_layer1,lean_set_sa_stacks1,lean_set_sa_stacks2,lean_set_sa_stacks3 \
use_papnet_loader=True \
batch_size=512 \
learning_rate=0.0005 \
epoch=30 \
workers=4 \
step_size=5 \
data_path={REPO_PATH}/datasets/partialmodelnet40/ \
checkpoint_path=best_model.pth \
kd.teacher.checkpoint_path={checkpoints_folder_path}/pct-p_p50.pth \
log_path={logs_path}

/content/pointcloud-bench/repos/Point-Transformers
mkdir: cannot create directory ‘/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/checkpoints/pct_lean’: File exists
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)
/content/pointcloud-bench/repos/Point-Transformers/train_kd.py:22: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespa